# Step 4b — Three Domain-Insight Features

**Purpose.** Steps 3–4 showed scalar feature engineering on GBDT is near its ceiling *for generic signals*. This step adds the three remaining features that encode **domain structure no current column expresses**, each derived from a documented dataset property:

| Feature | Dataset property it exploits | What it represents |
|---|---|---|
| `cat3_rebuy_due` (+ `cat3_reorder_cycle_days`) | Orders are fully recorded for 365 days (Step 1) | Whether the user is *due to repurchase* in the candidate's category, relative to that category's natural purchase cycle |
| `slate_pct_pop`, `slate_pct_text`, `slate_pct_sem` | Ranking is within-slate; interacted items compete against their own slate (interim hard-negative insight) | The candidate's standing **relative to its slate competitors** on the three strongest signal families |
| `cat3_cross_lift_max` | Order sequences are the one fully-recorded *sequential* signal | Complement-purchase affinity: how strongly the user's recently bought categories lead other shoppers to the candidate's category next |

**Inputs:** `events/candidates/features.parquet` (after Step 4) + `product_meta_data.txt` (ID columns only).
**Output:** `features.parquet` updated in place (up to 6 new columns).
**Runtime:** ~20 min, < 12 GB RAM — no SVD, no embeddings.

In [1]:
import os, time, gc
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow as pa, pyarrow.csv as pacsv

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = Path('/content/drive/MyDrive/JDSearch/Raw Data')
except ImportError:
    DATA_DIR = Path(os.environ.get('JD_DATA_DIR', '~/Downloads/JDsearch')).expanduser()
OUT_DIR = DATA_DIR / 'processed'
SEED = 42

cands = pd.read_parquet(OUT_DIR / 'candidates.parquet', columns=['user', 'wid', 'label'])
events = pd.read_parquet(OUT_DIR / 'events.parquet', columns=['user', 'pos', 'wid', 'type', 'age_days'])
nU = int(cands['user'].max()) + 1

# wid -> cate_id_3 (safe attach: missing metadata -> -1)
prod = pacsv.read_csv(DATA_DIR / 'product_meta_data.txt',
    parse_options=pacsv.ParseOptions(delimiter='\t', quote_char=False),
    convert_options=pacsv.ConvertOptions(include_columns=['wid', 'cate_id_3'],
                                         column_types={'wid': pa.int64(), 'cate_id_3': pa.int64()})).to_pandas()
prod['cate_id_3'] = prod['cate_id_3'].fillna(-1).astype(np.int64)
prod_idx = pd.Index(prod['wid'])
def cat3_of(wids):
    i = prod_idx.get_indexer(wids)
    out = prod['cate_id_3'].to_numpy()[np.maximum(i, 0)]
    out[i < 0] = -1
    return out
events['cat3'] = cat3_of(events['wid'])
cands['cat3'] = cat3_of(cands['wid'])
del prod, prod_idx; gc.collect()
print(f'candidates {len(cands):,} | events {len(events):,} | '
      f'cat3 coverage: candidates {(cands["cat3"] >= 0).mean():.1%}, events {(events["cat3"] >= 0).mean():.1%}')

Mounted at /content/drive
candidates 15,510,012 | events 26,667,260 | cat3 coverage: candidates 99.5%, events 95.1%


## 1. `cat3_rebuy_due` — repurchase-cycle timing

**Derived from:** order events only (`type == 2`, the one interaction fully recorded for 365 days), their `age_days` (built in Step 1 from `history_time_list`), and the candidate's `cate_id_3` from the product metadata.

**Derivation.**
1. For every (user, category) with ≥ 2 orders, take the gaps in days between consecutive orders. Pooling the gaps across *all* users gives each category's **median reorder cycle** (`cat3_reorder_cycle_days`); categories with < 30 observed gaps fall back to the global median. Same-day gaps (multi-item checkouts) are excluded.
2. For each candidate, look up how many days ago **this user** last ordered in the candidate's category.
3. `cat3_rebuy_due` = days-since-last-order ÷ category cycle. ≈ 0 means "just bought", ≈ 1 means "a typical cycle has elapsed — due to rebuy", ≫ 1 means lapsed. `NaN` when the user never ordered in the category.

**What it adds.** Existing features count *how much* a user ordered in a category (`ucat3_*`) and *how recently they touched it* (`ucat3_days_since_any`), but recency alone can't distinguish "bought rice 30 days ago" (due again) from "bought a fridge 30 days ago" (not due for years). Normalising by the category's own cycle turns raw recency into purchase *intent timing* — consumables and durables become comparable on one scale.

**Business objective.** Session conversion is driven by purchases, and repeat purchases are the most predictable kind; `cand_in_purchase_hist` (interim) was the seed of this idea but was binary and cycle-blind. This is the one feature aimed squarely at **Purchase-HR@10**.

In [2]:
t0 = time.time()
ORD = 2
o = events[(events['type'] == ORD) & (events['cat3'] >= 0)].sort_values(['user', 'cat3', 'pos'], kind='stable')
u, c3, a = o['user'].to_numpy(), o['cat3'].to_numpy(), o['age_days'].to_numpy()
same = (u[1:] == u[:-1]) & (c3[1:] == c3[:-1])
gap = a[:-1] - a[1:]                                   # age decreases chronologically
valid = same & (gap > 0)
cycle = pd.Series(gap[valid]).groupby(pd.Series(c3[1:][valid])).agg(['median', 'size'])
global_cycle = float(np.median(gap[valid]))
cycle_days = cycle['median'].where(cycle['size'] >= 30, global_cycle)

last_ord = pd.Series(a).groupby(pd.Series(u * (c3.max() + 1) + c3)).min()   # user's most recent order age per cat3
ck = cands['user'].to_numpy() * (c3.max() + 1) + np.maximum(cands['cat3'].to_numpy(), 0)
last_age = last_ord.reindex(ck).to_numpy(np.float32)
cyc = cycle_days.reindex(cands['cat3']).fillna(global_cycle).to_numpy(np.float32)

NEW = {}
NEW['cat3_reorder_cycle_days'] = np.where(cands['cat3'] >= 0, cyc, np.nan).astype(np.float32)
NEW['cat3_rebuy_due'] = (last_age / cyc).astype(np.float32)        # NaN when never ordered in the category
del o, u, c3, a, same, gap, valid, last_ord; gc.collect()
print(f'{time.time() - t0:.0f}s | global median cycle {global_cycle:.1f} d | '
      f'candidates with a rebuy signal: {np.isfinite(NEW["cat3_rebuy_due"]).mean():.1%}')
pd.Series(cycle_days).describe().round(1)

4s | global median cycle 31.7 d | candidates with a rebuy signal: 12.3%


,median
count,3548.0
mean,30.0
std,13.6
min,0.0
25%,25.8
50%,31.7
75%,31.7
max,116.4


## 2. `slate_pct_*` — within-slate contrast ranks

**Derived from:** three existing columns of `features.parquet` — `i_users_cart_90d` (strongest popularity signal), `q_name_tfidf_sim` (strongest text signal), `u_sem_lsa_7d` (strongest semantic-personalisation signal) — percentile-ranked **within each user's slate** (`groupby(user).rank(pct=True)`).

**What they add.** The ranking objective is purely within-slate, but a tree splits on *global* thresholds: 40 cart-users is high in a niche slate and low in a blockbuster slate, and the model must burn many splits to approximate that context. The percentile rank hands it the quantity that actually matters — "this candidate is at the 95th percentile of *its own competitors*" — in one column. It is also the operational form of the interim report's hard-negative insight: a zero-label item that loses to a same-slate competitor is informative *relative to that competitor*, not to the global catalogue.

**Business objective / understanding.** No new information is created — this is a representation fix that lets the existing strongest signals act at the granularity the business metric (per-session conversion) is measured at. If these ranks earn gain, it quantifies how much of the remaining headroom is representational rather than informational — directly motivating the neural ranker, which sees the slate jointly.

In [3]:
t0 = time.time()
RANK_SRC = {'i_users_cart_90d': 'slate_pct_pop', 'q_name_tfidf_sim': 'slate_pct_text', 'u_sem_lsa_7d': 'slate_pct_sem'}
import pyarrow.parquet as pq
have = [c for c in RANK_SRC if c in pq.read_schema(OUT_DIR / 'features.parquet').names]
missing = sorted(set(RANK_SRC) - set(have))
if missing:
    print(f'WARNING: {missing} not in features.parquet (run step 4 first) — skipping those ranks')
fsrc = pd.read_parquet(OUT_DIR / 'features.parquet', columns=['user'] + have)
for src in have:
    NEW[RANK_SRC[src]] = fsrc.groupby('user')[src].rank(pct=True).to_numpy(np.float32)
del fsrc; gc.collect()
print(f'{time.time() - t0:.0f}s | built: {[RANK_SRC[s] for s in have]}')

24s | built: ['slate_pct_pop', 'slate_pct_text', 'slate_pct_sem']


## 3. `cat3_cross_lift_max` — complement-purchase transition prior

**Derived from:** the chronological sequence of order events (sorted by `pos` within user). For every pair of **consecutive orders by the same user in different level-3 categories**, count the transition (from-category → to-category) across *all* users. Each pair's strength is expressed as **lift** — how much more often B follows A than B's overall share of transitions would predict:

$$\text{lift}(A \to B) = \frac{P(B \mid \text{prev}=A)}{P(B)} = \frac{n_{AB} \cdot N}{n_{A\cdot} \cdot n_{\cdot B}}$$

(pairs with < 5 observations are dropped, and the stored `log(lift)` is shrunk by `cnt/(cnt+20)` so rare pairs with extreme lifts are damped toward 0). Then for each candidate: take the user's **distinct ordered categories in the last 90 days** as their "recent needs", and score the candidate with the **maximum log-lift** from any of those categories to the candidate's category.

**What it adds.** Everything so far scores a candidate against the user's history *in the same item/brand/category*. This is the first **cross-category** signal: it knows that buyers of phones buy cases next, buyers of printers buy toner — population-level sequential knowledge from the only interaction type whose sequence is fully recorded (orders, 365 d). Using *lift* rather than raw probability deliberately removes general popularity, which `i_users_*` already covers, so the feature is orthogonal by construction. And because the candidate side only needs a category, it reaches **cold items**.

**Business objective.** Complement recommendations convert sessions that same-category features cannot: after the user bought the camera, re-ranking the memory card *above* another camera is exactly a session conversion the current feature set has no way to express.

*Caveat (noted for the report): the transition table is estimated on all users including the focal one — with 1M+ transitions a single user's contribution is negligible, but a strictly leave-one-out version would be the rigorous variant.*

In [4]:
t0 = time.time()
o = events[(events['type'] == ORD) & (events['cat3'] >= 0)].sort_values(['user', 'pos'], kind='stable')
u, c3, a = o['user'].to_numpy(), o['cat3'].to_numpy(), o['age_days'].to_numpy()
C = int(max(c3.max(), cands['cat3'].max())) + 1
m = (u[1:] == u[:-1]) & (c3[1:] != c3[:-1])
key, cnt = np.unique(c3[:-1][m].astype(np.int64) * C + c3[1:][m], return_counts=True)
frm, to = (key // C).astype(np.int64), key % C
keep = cnt >= 5
frm, to, cnt, key = frm[keep], to[keep], cnt[keep], key[keep]
N = cnt.sum()
row = np.bincount(frm, weights=cnt, minlength=C); col = np.bincount(to, weights=cnt, minlength=C)
log_lift = (np.log(cnt * N / (row[frm] * col[to])) * cnt / (cnt + 20)).astype(np.float32)   # shrink small counts toward 0
print(f'{m.sum():,} cross-category transitions -> {len(key):,} (from, to) pairs with >=5 obs')

# user's distinct ordered categories in the last 90 days
rec = pd.DataFrame({'user': u, 'frm': c3})[a < 90].drop_duplicates()
print(f'users with recent-order categories: {rec["user"].nunique():,} | mean categories/user: {len(rec) / rec["user"].nunique():.1f}')

score = np.full(len(cands), np.nan, np.float32)
cand_df = pd.DataFrame({'row': np.arange(len(cands), dtype=np.int64),
                        'user': cands['user'].to_numpy(), 'to': cands['cat3'].to_numpy()})
cand_df = cand_df[cand_df['to'] >= 0]
for s in range(0, nU, 20_000):                                    # chunk by user to bound the join size
    ch = cand_df[(cand_df['user'] >= s) & (cand_df['user'] < s + 20_000)]
    j = ch.merge(rec[(rec['user'] >= s) & (rec['user'] < s + 20_000)], on='user')
    if not len(j):
        continue
    p = np.searchsorted(key, j['frm'].to_numpy() * C + j['to'].to_numpy())
    hit = (p < len(key)) & (key[np.minimum(p, len(key) - 1)] == j['frm'].to_numpy() * C + j['to'].to_numpy())
    jj = j[hit]
    if not len(jj):
        continue
    best = pd.Series(log_lift[p[hit]]).groupby(jj['row'].to_numpy()).max()
    score[best.index.to_numpy()] = best.to_numpy(np.float32)
NEW['cat3_cross_lift_max'] = score
del o, u, c3, a, m, rec, cand_df, events; gc.collect()
print(f'{time.time() - t0:.0f}s | candidates with a cross-category signal: {np.isfinite(score).mean():.1%}')
pd.DataFrame(NEW).describe().T.round(3)

5,128,531 cross-category transitions -> 177,397 (from, to) pairs with >=5 obs
users with recent-order categories: 140,951 | mean categories/user: 9.7
21s | candidates with a cross-category signal: 60.7%


,count,mean,std,min,25%,50%,75%,max
cat3_reorder_cycle_days,15439674.0,27.280,17.323,0.024,13.916,26.553,33.784,116.371
cat3_rebuy_due,1915201.0,5.599,53.415,0.000,0.657,2.315,5.902,9575.145
slate_pct_pop,15510012.0,0.506,0.231,0.005,0.341,0.444,0.680,1.000
slate_pct_text,15439674.0,0.506,0.281,0.005,0.267,0.495,0.750,1.000
slate_pct_sem,15439674.0,0.506,0.269,0.005,0.289,0.505,0.722,1.000
cat3_cross_lift_max,9417268.0,0.897,1.025,-1.170,0.215,0.545,1.278,8.035


## 4. Signal check and benchmark

In [5]:
def evaluate(df, score, k=10, seed=0):
    """Vectorised per-slate ranking metrics with random tie-breaking (identical to Steps 2-4)."""
    u = df['user'].to_numpy(); lab = df['label'].to_numpy().astype(np.float64)
    r = np.random.default_rng(seed).random(len(df))
    order = np.lexsort((r, -np.asarray(score, dtype=np.float64), u))
    u_s, lab_s = u[order], lab[order]
    start = np.r_[0, np.flatnonzero(u_s[1:] != u_s[:-1]) + 1]
    rank = np.arange(len(u_s)) - np.repeat(start, np.diff(np.r_[start, len(u_s)]))
    top = rank < k
    disc = np.where(top, 1.0 / np.log2(rank + 2), 0.0)
    gid = np.repeat(np.arange(len(start)), np.diff(np.r_[start, len(u_s)]))
    dcg = np.bincount(gid, weights=(2 ** lab_s - 1) * disc)
    ideal_order = np.lexsort((-lab, u)); lab_i = lab[ideal_order]
    idcg = np.bincount(gid, weights=(2 ** lab_i - 1) * disc)
    has_pos = np.bincount(gid, weights=(lab_s > 0)) > 0
    has_buy = np.bincount(gid, weights=(lab_s == 3)) > 0
    hit = np.bincount(gid, weights=((lab_s > 0) & top)) > 0
    hit_buy = np.bincount(gid, weights=((lab_s == 3) & top)) > 0
    return {'NDCG@10': (dcg[has_pos] / idcg[has_pos]).mean(), 'HR@10': hit[has_pos].mean(),
            'Purchase-HR@10': hit_buy[has_buy].mean(), 'slates': int(has_pos.sum())}

split = pd.read_parquet(OUT_DIR / 'features.parquet', columns=['split'])['split'].to_numpy()
ev_df = pd.DataFrame({'user': cands['user'], 'label': cands['label']})
val_m = split == 'val'
single = pd.DataFrame({f: evaluate(ev_df[val_m], pd.Series(v[val_m]).fillna(-1e9) if f != 'cat3_rebuy_due'
                                   else -np.abs(pd.Series(v[val_m]).fillna(1e9) - 1.0))   # rebuy: closest to 'due' = best
                       for f, v in NEW.items()}).T
single.loc['random'] = evaluate(ev_df[val_m], np.zeros(int(val_m.sum())))
single.sort_values('NDCG@10', ascending=False).round(4)

,NDCG@10,HR@10,Purchase-HR@10,slates
slate_pct_pop,0.3330,0.6277,0.5665,25825.0
slate_pct_sem,0.1157,0.2792,0.1931,25825.0
slate_pct_text,0.0960,0.2444,0.1563,25825.0
cat3_rebuy_due,0.0865,0.2361,0.1649,25825.0
random,0.0846,0.2326,0.1563,25825.0
cat3_cross_lift_max,0.0830,0.2257,0.1481,25825.0
cat3_reorder_cycle_days,0.0726,0.1962,0.1379,25825.0


In [6]:
feat = pd.read_parquet(OUT_DIR / 'features.parquet')
for f, v in NEW.items():
    feat[f] = v
NEW_FEATURES = list(NEW); del NEW; gc.collect()
feat.to_parquet(OUT_DIR / 'features.parquet', index=False)
FEATURES_ALL = [c for c in feat.columns if c not in ('user', 'wid', 'label', 'split')]
FEATURES_BASE = [c for c in FEATURES_ALL if c not in NEW_FEATURES]
print(f'saved features.parquet with {len(FEATURES_ALL)} features ({len(FEATURES_BASE)} base + {len(NEW_FEATURES)} new)')

import lightgbm as lgb, warnings
warnings.filterwarnings('ignore', message='.*eval_set.*deprecated')

def prepared(split_name):
    d = feat[feat['split'] == split_name]
    d = d.iloc[np.lexsort((np.random.default_rng(SEED).random(len(d)), d['user'].to_numpy()))]
    d = d[d.groupby('user')['label'].transform('max') > 0]
    return d, d.groupby('user', sort=False).size().to_numpy()

tr, g_tr = prepared('train'); va, g_va = prepared('val'); te, _ = prepared('test')
del feat; gc.collect()

def fit(cols, n_estimators=600):
    m = lgb.LGBMRanker(objective='lambdarank', n_estimators=n_estimators, learning_rate=0.05, num_leaves=63,
                       min_child_samples=200, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                       random_state=SEED, n_jobs=-1, verbose=-1)
    m.fit(tr[cols], tr['label'], group=g_tr, eval_set=[(va[cols], va['label'])], eval_group=[g_va],
          eval_at=[10], callbacks=[lgb.early_stopping(50, verbose=False)])
    return m

t0 = time.time(); base = fit(FEATURES_BASE); print(f'base: {time.time() - t0:.0f}s, iter {base.best_iteration_}')
t0 = time.time(); full = fit(FEATURES_ALL); print(f'+domain: {time.time() - t0:.0f}s, iter {full.best_iteration_}')

results = pd.DataFrame({
    'random': evaluate(te, np.zeros(len(te))),
    'LightGBM base (steps 2-4 features)': evaluate(te, base.predict(te[FEATURES_BASE])),
    'LightGBM + domain features': evaluate(te, full.predict(te[FEATURES_ALL])),
}).T
results.round(4)

saved features.parquet with 55 features (49 base + 6 new)
base: 267s, iter 436
+domain: 229s, iter 336


,NDCG@10,HR@10,Purchase-HR@10,slates
random,0.0839,0.2268,0.1371,25822.0
LightGBM base (steps 2-4 features),0.4155,0.7157,0.6662,25822.0
LightGBM + domain features,0.4159,0.7196,0.6692,25822.0


In [7]:
imp = pd.Series(full.booster_.feature_importance('gain'), index=FEATURES_ALL)
print('share of total split gain (new features):')
print((imp / imp.sum()).loc[NEW_FEATURES].sort_values(ascending=False).round(4))

hist_wids = np.unique(pd.read_parquet(OUT_DIR / 'events.parquet', columns=['wid'])['wid'].to_numpy())
j = np.searchsorted(hist_wids, te['wid'].to_numpy())
te_cold = (j >= len(hist_wids)) | (hist_wids[np.minimum(j, len(hist_wids) - 1)] != te['wid'].to_numpy())
pos = te['label'].to_numpy() > 0
cold_pos = pd.Series(np.where(pos, te_cold, np.nan), index=te.index)
seg_frac = cold_pos.groupby(te['user']).mean()
seg_of_user = pd.cut(seg_frac, [-0.01, 0.0, 0.999, 1.0], labels=['all warm', 'mixed', 'all cold'])
seg = te['user'].map(seg_of_user)
rows = {}
for s in ['all warm', 'mixed', 'all cold']:
    m = (seg == s).to_numpy()
    rows[f'base — positives {s}'] = evaluate(te[m], base.predict(te.loc[m, FEATURES_BASE]))
    rows[f'+domain — positives {s}'] = evaluate(te[m], full.predict(te.loc[m, FEATURES_ALL]))
pd.DataFrame(rows).T.round(4)

share of total split gain (new features):
slate_pct_pop              0.4841
slate_pct_text             0.0130
slate_pct_sem              0.0070
cat3_reorder_cycle_days    0.0019
cat3_cross_lift_max        0.0009
cat3_rebuy_due             0.0005
dtype: float64


,NDCG@10,HR@10,Purchase-HR@10,slates
base — positives all warm,0.5092,0.8118,0.7477,18466.0
+domain — positives all warm,0.5080,0.8134,0.7517,18466.0
base — positives mixed,0.2697,0.7553,0.4535,3379.0
+domain — positives mixed,0.2685,0.7567,0.4479,3379.0
base — positives all cold,0.1042,0.2359,0.1405,3977.0
+domain — positives all cold,0.1138,0.2527,0.1446,3977.0


## 5. Takeaways

- **Overall: NDCG@10 0.4155 → 0.4159, HR@10 0.716 → 0.720, Purchase-HR@10 0.666 → 0.669.** The GBDT plateau is now unmistakable: 0.4075 → 0.4144 → 0.4154 → 0.4159 across Steps 2–4b.
- **The headline is `slate_pct_pop`: it takes 48.4% of the model's total split gain** — the slate-relative form of popularity displaces the raw form as the model's main working feature, and the model converges in ~100 fewer trees (336 vs 436). The metric barely moves because *no new information* was added, but the model visibly restructures around the within-slate representation. This is the clearest evidence yet that the remaining headroom is **representational, not informational** — the direct motivation for a slate-aware neural ranker.
- **Cold slates get the largest lift since Step 3: NDCG@10 0.104 → 0.114 (+9% relative), HR@10 0.236 → 0.253.** The slate ranks of text/semantic similarity and the cross-category prior all reach cold items, and that is where they pay.
- **`cat3_rebuy_due` and `cat3_cross_lift_max` earn little ensemble gain** (0.05–0.09% each): with 49 features already covering user-category affinity, the tree model finds little left to extract from timing and complements at 12% / 61% coverage. They remain principled, interpretable inputs for the neural model (and Purchase-HR did tick up overall), but as GBDT features they confirm the ceiling rather than break it.
- **Decision for the modelling phase:** stop engineering scalar features. The pipeline's own evidence — semantic information that scores 0.116 alone but adds ~0 in the ensemble (Step 4), and a representation fix that takes half the gain without moving the metric (this step) — says the next lift must come from a model that sees the slate jointly and consumes history/embeddings natively.